# SAM3 Few-Shot Metric Conversion

This notebook converts SAM3 few-shot COCO instance predictions into dense binary masks so they can be compared with U-Net/ResNet34 using the same pixel-level metrics.

Workflow:

1. Read each run's `test/_annotations.coco.json` ground truth.
2. Read each run's `dumps/ttd/test/coco_predictions_segm.json` predictions.
3. For each image, union all SAM3 predicted instance masks above a score threshold.
4. Compute pixel-level IoU, F1, precision, and recall.
5. Report all-image, positive-only, negative-only, and global pixel metrics.

Run this after SAM3 train/eval jobs have produced prediction JSON files.


In [8]:
import os
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd


def find_project_repo_root(start=None):
    """Find the CSCI5527-final repo root from the current notebook working directory."""
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "SAM3").exists() and (candidate / "TACK_Tunnel_Data").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the CSCI5527-final repo root. "
        "Start Jupyter inside the project repo, or update this helper."
    )


PROJECT_REPO_ROOT = find_project_repo_root()
WORKSPACE_ROOT = PROJECT_REPO_ROOT.parent
PROJECT_ROOT = WORKSPACE_ROOT  # Backward-compatible name used below.
SAM3_WORK_ROOT = PROJECT_REPO_ROOT / "SAM3"
SAM3_REPO_ROOT = Path(os.environ.get("SAM3_REPO_ROOT", WORKSPACE_ROOT / "sam3")).resolve()

EXPERIMENT_GROUPS = {
    "single_tb": "Single-TB",
    "shift_ta_tc_to_tb_10pct": "Shift-TA_TC-to-TB_10pct",
    "shift_ta_tb_to_tc_10pct": "Shift-TA_TB-to-TC_10pct",
}

# Keep the same order as the batch training notebook: Single-TB first, then shift experiments.
BATCH_GROUP_ORDER = [
    "single_tb",
    "shift_ta_tc_to_tb_10pct",
    "shift_ta_tb_to_tc_10pct",
]
BATCH_SHOTS = [5, 10, 25, 50]
RUN_TAG = "stable_lowlr_v1"

# Threshold sweep. You can add/remove values here.
# SCORE_THRESHOLDS = [0.0, 0.01, 0.02, 0.05, 0.10, 0.20, 0.30, 0.50]
SCORE_THRESHOLDS = [0.3]
OUTPUT_DIR = SAM3_WORK_ROOT / "outputs" / "fewshot_metric_conversion"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = []
for group_key in BATCH_GROUP_ORDER:
    experiment_name = EXPERIMENT_GROUPS[group_key]
    for shot in BATCH_SHOTS:
        key = f"{group_key}_{shot}shot_{RUN_TAG}"
        run_root = SAM3_WORK_ROOT / "fewshot_data" / experiment_name / f"{shot}_shot_per_class"
        run_dir = SAM3_WORK_ROOT / "outputs" / "fewshot_runs" / experiment_name / f"{shot}_shot_per_class_{RUN_TAG}"
        EXPERIMENTS.append({
            "experiment_key": key,
            "experiment_name": experiment_name,
            "shot_per_class": shot,
            "run_tag": RUN_TAG,
            "gt_json": run_root / "test" / "_annotations.coco.json",
            "pred_json": run_dir / "dumps" / "ttd" / "test" / "coco_predictions_segm.json",
        })

pd.DataFrame([
    {
        "experiment_key": e["experiment_key"],
        "experiment_name": e["experiment_name"],
        "shot_per_class": e["shot_per_class"],
        "gt_exists": e["gt_json"].exists(),
        "pred_exists": e["pred_json"].exists(),
        "pred_json": str(e["pred_json"]),
    }
    for e in EXPERIMENTS
])


,experiment_key,experiment_name,shot_per_class,gt_exists,pred_exists,pred_json
0,single_tb_5shot_stable_lowlr_v1,Single-TB,5,True,True,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
1,single_tb_10shot_stable_lowlr_v1,Single-TB,10,True,True,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
2,single_tb_25shot_stable_lowlr_v1,Single-TB,25,True,True,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
3,single_tb_50shot_stable_lowlr_v1,Single-TB,50,True,True,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
4,shift_ta_tc_to_tb_10pct_5shot_stable_lowlr_v1,Shift-TA_TC-to-TB_10pct,5,True,True,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
5,shift_ta_tc_to_tb_10pct_10shot_stable_lowlr_v1,Shift-TA_TC-to-TB_10pct,10,True,True,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
6,shift_ta_tc_to_tb_10pct_25shot_stable_lowlr_v1,Shift-TA_TC-to-TB_10pct,25,True,True,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
7,shift_ta_tc_to_tb_10pct_50shot_stable_lowlr_v1,Shift-TA_TC-to-TB_10pct,50,True,True,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
8,shift_ta_tb_to_tc_10pct_5shot_stable_lowlr_v1,Shift-TA_TB-to-TC_10pct,5,True,True,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
9,shift_ta_tb_to_tc_10pct_10shot_stable_lowlr_v1,Shift-TA_TB-to-TC_10pct,10,True,True,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...


In [9]:
def _decode_compressed_rle_counts(counts_text):
    """Decode COCO compressed RLE counts into run lengths.

    This avoids requiring pycocotools just for metric conversion. It supports the
    compressed RLE strings used in the SAM3 prediction and GT JSON files.
    """
    if isinstance(counts_text, list):
        return counts_text
    if isinstance(counts_text, bytes):
        counts_text = counts_text.decode("ascii")

    counts = []
    pos = 0
    count_index = 0
    while pos < len(counts_text):
        value = 0
        shift = 0
        while True:
            char_value = ord(counts_text[pos]) - 48
            pos += 1
            value |= (char_value & 0x1F) << shift
            shift += 5
            if not (char_value & 0x20):
                if char_value & 0x10:
                    value |= -1 << shift
                break
        if count_index > 2:
            value += counts[count_index - 2]
        counts.append(value)
        count_index += 1
    return counts


def decode_coco_rle(rle):
    """Decode a COCO RLE object into a boolean H x W mask."""
    height, width = rle["size"]
    counts = _decode_compressed_rle_counts(rle["counts"])
    flat = np.zeros(height * width, dtype=bool)
    index = 0
    value = False
    for run_length in counts:
        if run_length < 0:
            raise ValueError(f"Invalid negative RLE run length: {run_length}")
        if value and run_length:
            flat[index:index + run_length] = True
        index += run_length
        value = not value
    return flat.reshape((height, width), order="F")


def pixel_stats(pred_mask, true_mask, smooth=1e-6):
    pred = pred_mask.astype(bool)
    true = true_mask.astype(bool)

    tp = float(np.logical_and(pred, true).sum())
    fp = float(np.logical_and(pred, np.logical_not(true)).sum())
    fn = float(np.logical_and(np.logical_not(pred), true).sum())
    tn = float(np.logical_and(np.logical_not(pred), np.logical_not(true)).sum())

    iou = (tp + smooth) / (tp + fp + fn + smooth)
    precision = (tp + smooth) / (tp + fp + smooth)
    recall = (tp + smooth) / (tp + fn + smooth)
    f1 = 2.0 * precision * recall / (precision + recall + smooth)

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "iou": iou,
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }


def safe_mean(values):
    values = [v for v in values if not pd.isna(v)]
    return float(np.mean(values)) if values else np.nan


def summarize_per_image(per_image_df):
    positive = per_image_df[per_image_df["has_gt_crack"]]
    negative = per_image_df[~per_image_df["has_gt_crack"]]

    total_tp = per_image_df["tp"].sum()
    total_fp = per_image_df["fp"].sum()
    total_fn = per_image_df["fn"].sum()
    global_iou = (total_tp + 1e-6) / (total_tp + total_fp + total_fn + 1e-6)
    global_precision = (total_tp + 1e-6) / (total_tp + total_fp + 1e-6)
    global_recall = (total_tp + 1e-6) / (total_tp + total_fn + 1e-6)
    global_f1 = 2 * global_precision * global_recall / (global_precision + global_recall + 1e-6)

    return {
        "num_test": int(len(per_image_df)),
        "num_positive": int(len(positive)),
        "num_negative": int(len(negative)),
        "test_iou": safe_mean(per_image_df["iou"]),
        "test_f1": safe_mean(per_image_df["f1"]),
        "test_precision": safe_mean(per_image_df["precision"]),
        "test_recall": safe_mean(per_image_df["recall"]),
        "positive_iou": safe_mean(positive["iou"]),
        "positive_f1": safe_mean(positive["f1"]),
        "positive_precision": safe_mean(positive["precision"]),
        "positive_recall": safe_mean(positive["recall"]),
        "negative_clean_rate": float((negative["pred_pixels"] == 0).mean()) if len(negative) else np.nan,
        "negative_fp_rate": float((negative["pred_pixels"] > 0).mean()) if len(negative) else np.nan,
        "mean_num_predictions": safe_mean(per_image_df["num_predictions"]),
        "mean_gt_pixels": safe_mean(per_image_df["gt_pixels"]),
        "mean_pred_pixels": safe_mean(per_image_df["pred_pixels"]),
        "global_iou": float(global_iou),
        "global_f1": float(global_f1),
        "global_precision": float(global_precision),
        "global_recall": float(global_recall),
    }


In [10]:
def load_ground_truth_masks(gt_json_path):
    gt = json.loads(Path(gt_json_path).read_text())
    image_info = {image["id"]: image for image in gt["images"]}
    gt_masks = {
        image_id: np.zeros((image["height"], image["width"]), dtype=bool)
        for image_id, image in image_info.items()
    }

    for ann in gt.get("annotations", []):
        segmentation = ann.get("segmentation")
        if not isinstance(segmentation, dict):
            raise NotImplementedError(
                "This notebook expects RLE segmentations. If your JSON uses polygons, "
                "run it in an environment with pycocotools and add polygon conversion."
            )
        gt_masks[ann["image_id"]] |= decode_coco_rle(segmentation)

    return gt, image_info, gt_masks


def convert_one_experiment(experiment, score_threshold):
    gt_json_path = experiment["gt_json"]
    pred_json_path = experiment["pred_json"]

    if not gt_json_path.exists():
        raise FileNotFoundError(f"Missing ground-truth JSON: {gt_json_path}")
    if not pred_json_path.exists() or pred_json_path.stat().st_size <= 2:
        raise FileNotFoundError(f"Missing or empty prediction JSON: {pred_json_path}")

    _, image_info, gt_masks = load_ground_truth_masks(gt_json_path)
    pred_masks = {image_id: np.zeros_like(mask) for image_id, mask in gt_masks.items()}
    num_predictions = {image_id: 0 for image_id in gt_masks}

    predictions = json.loads(pred_json_path.read_text())
    for pred_ann in predictions:
        image_id = pred_ann.get("image_id")
        if image_id not in pred_masks:
            continue
        if float(pred_ann.get("score", 0.0)) < score_threshold:
            continue
        segmentation = pred_ann.get("segmentation")
        if not isinstance(segmentation, dict):
            continue
        pred_masks[image_id] |= decode_coco_rle(segmentation)
        num_predictions[image_id] += 1

    rows = []
    for image_id in sorted(gt_masks):
        gt_mask = gt_masks[image_id]
        pred_mask = pred_masks[image_id]
        stats = pixel_stats(pred_mask, gt_mask)
        image = image_info[image_id]
        rows.append({
            "experiment_key": experiment["experiment_key"],
            "experiment_name": experiment["experiment_name"],
            "shot_per_class": experiment["shot_per_class"],
            "run_tag": experiment["run_tag"],
            "score_threshold": score_threshold,
            "image_id": image_id,
            "file_name": image.get("file_name", ""),
            "has_gt_crack": bool(gt_mask.any()),
            "has_prediction": bool(pred_mask.any()),
            "gt_pixels": int(gt_mask.sum()),
            "pred_pixels": int(pred_mask.sum()),
            "num_predictions": int(num_predictions[image_id]),
            **stats,
        })

    per_image_df = pd.DataFrame(rows)
    summary = summarize_per_image(per_image_df)
    summary.update({
        "experiment_key": experiment["experiment_key"],
        "experiment_name": experiment["experiment_name"],
        "shot_per_class": experiment["shot_per_class"],
        "run_tag": experiment["run_tag"],
        "score_threshold": score_threshold,
        "gt_json": str(gt_json_path),
        "pred_json": str(pred_json_path),
    })
    return summary, per_image_df

print("Conversion helpers ready.")


Conversion helpers ready.


In [11]:
summary_rows = []
per_image_frames = []
skipped_rows = []

for experiment in EXPERIMENTS:
    print("\n" + "=" * 100)
    print(f"Converting {experiment['experiment_key']}")
    print("Prediction JSON:", experiment["pred_json"])

    if not experiment["pred_json"].exists() or experiment["pred_json"].stat().st_size <= 2:
        print("Skipping: prediction JSON does not exist yet or is empty.")
        skipped_rows.append({
            "experiment_key": experiment["experiment_key"],
            "reason": "missing_or_empty_prediction_json",
            "pred_json": str(experiment["pred_json"]),
        })
        continue

    for score_threshold in SCORE_THRESHOLDS:
        summary, per_image_df = convert_one_experiment(experiment, score_threshold)
        summary_rows.append(summary)
        per_image_frames.append(per_image_df)
        print(
            f"threshold={score_threshold:.2f} "
            f"test_iou={summary['test_iou']:.4f} "
            f"test_f1={summary['test_f1']:.4f} "
            f"positive_iou={summary['positive_iou']:.4f} "
            f"positive_f1={summary['positive_f1']:.4f} "
            f"negative_fp_rate={summary['negative_fp_rate']:.4f}"
        )

summary_df = pd.DataFrame(summary_rows)
per_image_df = pd.concat(per_image_frames, ignore_index=True) if per_image_frames else pd.DataFrame()
skipped_df = pd.DataFrame(skipped_rows)

summary_csv = OUTPUT_DIR / "sam3_fewshot_pixel_metric_summary_threshold_sweep.csv"
per_image_csv = OUTPUT_DIR / "sam3_fewshot_pixel_metric_per_image_threshold_sweep.csv"
skipped_csv = OUTPUT_DIR / "sam3_fewshot_pixel_metric_skipped_runs.csv"

summary_df.to_csv(summary_csv, index=False)
per_image_df.to_csv(per_image_csv, index=False)
skipped_df.to_csv(skipped_csv, index=False)

print("\nSaved summary:", summary_csv)
print("Saved per-image metrics:", per_image_csv)
print("Saved skipped-runs list:", skipped_csv)

display(summary_df)
if len(skipped_df):
    print("Skipped runs:")
    display(skipped_df)



Converting single_tb_5shot_stable_lowlr_v1
Prediction JSON: /users/7/yu001011/csci5527/CSCI5527-final/SAM3/outputs/fewshot_runs/Single-TB/5_shot_per_class_stable_lowlr_v1/dumps/ttd/test/coco_predictions_segm.json
threshold=0.30 test_iou=0.4697 test_f1=0.5110 positive_iou=0.1478 positive_f1=0.2303 negative_fp_rate=0.2083

Converting single_tb_10shot_stable_lowlr_v1
Prediction JSON: /users/7/yu001011/csci5527/CSCI5527-final/SAM3/outputs/fewshot_runs/Single-TB/10_shot_per_class_stable_lowlr_v1/dumps/ttd/test/coco_predictions_segm.json
threshold=0.30 test_iou=0.4754 test_f1=0.5190 positive_iou=0.1591 positive_f1=0.2464 negative_fp_rate=0.2083

Converting single_tb_25shot_stable_lowlr_v1
Prediction JSON: /users/7/yu001011/csci5527/CSCI5527-final/SAM3/outputs/fewshot_runs/Single-TB/25_shot_per_class_stable_lowlr_v1/dumps/ttd/test/coco_predictions_segm.json
threshold=0.30 test_iou=0.5483 test_f1=0.5936 positive_iou=0.1800 positive_f1=0.2705 negative_fp_rate=0.0833

Converting single_tb_50sho

,num_test,num_positive,num_negative,test_iou,test_f1,test_precision,test_recall,positive_iou,positive_f1,positive_precision,...,global_f1,global_precision,global_recall,experiment_key,experiment_name,shot_per_class,run_tag,score_threshold,gt_json,pred_json
0,48,24,24,0.469736,0.510962,0.534374,0.861448,0.147805,0.230257,0.277082,...,0.269831,0.159009,0.890412,single_tb_5shot_stable_lowlr_v1,Single-TB,5,stable_lowlr_v1,0.3,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
1,48,24,24,0.475382,0.519027,0.561204,0.862115,0.159098,0.246388,0.330742,...,0.294468,0.176341,0.891986,single_tb_10shot_stable_lowlr_v1,Single-TB,10,stable_lowlr_v1,0.3,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
2,48,24,24,0.548328,0.593588,0.698537,0.816334,0.179990,0.270510,0.480406,...,0.372171,0.241941,0.806048,single_tb_25shot_stable_lowlr_v1,Single-TB,25,stable_lowlr_v1,0.3,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
3,48,24,24,0.540708,0.590059,0.695001,0.799704,0.206416,0.305118,0.515002,...,0.427920,0.296238,0.770355,single_tb_50shot_stable_lowlr_v1,Single-TB,50,stable_lowlr_v1,0.3,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
4,48,24,24,0.470343,0.512004,0.534993,0.861391,0.149020,0.232342,0.278319,...,0.279211,0.165584,0.889829,shift_ta_tc_to_tb_10pct_5shot_stable_lowlr_v1,Shift-TA_TC-to-TB_10pct,5,stable_lowlr_v1,0.3,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
5,48,24,24,0.475069,0.518272,0.540193,0.861744,0.158470,0.244877,0.288720,...,0.285517,0.169933,0.892716,shift_ta_tc_to_tb_10pct_10shot_stable_lowlr_v1,Shift-TA_TC-to-TB_10pct,10,stable_lowlr_v1,0.3,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
6,48,24,24,0.522319,0.566001,0.629740,0.837142,0.169639,0.257002,0.384479,...,0.319754,0.198936,0.814301,shift_ta_tc_to_tb_10pct_25shot_stable_lowlr_v1,Shift-TA_TC-to-TB_10pct,25,stable_lowlr_v1,0.3,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
7,48,24,24,0.536817,0.585902,0.668467,0.826053,0.198634,0.296804,0.461933,...,0.386669,0.256117,0.788697,shift_ta_tc_to_tb_10pct_50shot_stable_lowlr_v1,Shift-TA_TC-to-TB_10pct,50,stable_lowlr_v1,0.3,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
8,38,19,19,0.549616,0.575344,0.876896,0.640908,0.151863,0.203320,0.806423,...,0.371636,0.306909,0.470964,shift_ta_tb_to_tc_10pct_5shot_stable_lowlr_v1,Shift-TA_TB-to-TC_10pct,5,stable_lowlr_v1,0.3,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...
9,38,19,19,0.551037,0.576743,0.881234,0.635971,0.154706,0.206117,0.815101,...,0.374805,0.318896,0.454487,shift_ta_tb_to_tc_10pct_10shot_stable_lowlr_v1,Shift-TA_TB-to-TC_10pct,10,stable_lowlr_v1,0.3,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...


## Pick One Threshold Per Experiment

For the final report, avoid selecting the best threshold directly on the test set if you want a strict experimental protocol. A good default is:

- Use one fixed threshold for all SAM3 runs, such as `0.10` or `0.20`, or
- Select threshold using validation predictions, then apply that threshold to test.

The cell below is a convenience view that chooses the best threshold by `positive_f1` from the test sweep. Use it for exploration, not as the strict final result unless you clearly state that threshold was selected post hoc.


In [12]:
if summary_df.empty:
    raise RuntimeError("Run the threshold-sweep conversion cell first.")

# best_by_positive_f1 = (
#     summary_df.sort_values(["experiment_key", "positive_f1", "positive_iou"], ascending=[True, False, False])
#     .groupby("experiment_key", as_index=False)
#     .head(1)
#     .sort_values(["experiment_name", "shot_per_class"])
# )

FIXED_SCORE_THRESHOLD = 0.30

fixed_threshold_df = (
    summary_df[summary_df["score_threshold"] == FIXED_SCORE_THRESHOLD]
    .sort_values(["experiment_name", "shot_per_class"])
    .copy()
)

best_by_positive_f1 = fixed_threshold_df

best_csv = OUTPUT_DIR / "sam3_fewshot_pixel_metric_best_test_positive_f1.csv"
best_by_positive_f1.to_csv(best_csv, index=False)
print("Saved exploratory best-threshold table:", best_csv)

display(best_by_positive_f1[[
    "experiment_name",
    "shot_per_class",
    "score_threshold",
    "test_iou",
    "test_f1",
    "test_precision",
    "test_recall",
    "positive_iou",
    "positive_f1",
    "positive_precision",
    "positive_recall",
    "negative_fp_rate",
    "mean_num_predictions",
]])


Saved exploratory best-threshold table: /users/7/yu001011/csci5527/CSCI5527-final/SAM3/outputs/fewshot_metric_conversion/sam3_fewshot_pixel_metric_best_test_positive_f1.csv


,experiment_name,shot_per_class,score_threshold,test_iou,test_f1,test_precision,test_recall,positive_iou,positive_f1,positive_precision,positive_recall,negative_fp_rate,mean_num_predictions
8,Shift-TA_TB-to-TC_10pct,5,0.3,0.549616,0.575344,0.876896,0.640908,0.151863,0.203320,0.806423,0.281815,0.052632,0.289474
9,Shift-TA_TB-to-TC_10pct,10,0.3,0.551037,0.576743,0.881234,0.635971,0.154706,0.206117,0.815101,0.271943,0.052632,0.263158
10,Shift-TA_TB-to-TC_10pct,25,0.3,0.525805,0.546028,0.914113,0.578445,0.104241,0.144688,0.880858,0.156890,0.052632,0.210526
11,Shift-TA_TB-to-TC_10pct,50,0.3,0.550985,0.571756,0.944274,0.572398,0.101970,0.143512,0.888549,0.144795,0.000000,0.157895
4,Shift-TA_TC-to-TB_10pct,5,0.3,0.470343,0.512004,0.534993,0.861391,0.149020,0.232342,0.278319,0.722782,0.208333,2.979167
5,Shift-TA_TC-to-TB_10pct,10,0.3,0.475069,0.518272,0.540193,0.861744,0.158470,0.244877,0.288720,0.723488,0.208333,3.104167
6,Shift-TA_TC-to-TB_10pct,25,0.3,0.522319,0.566001,0.629740,0.837142,0.169639,0.257002,0.384479,0.674285,0.125000,2.958333
7,Shift-TA_TC-to-TB_10pct,50,0.3,0.536817,0.585902,0.668467,0.826053,0.198634,0.296804,0.461933,0.652106,0.125000,2.645833
0,Single-TB,5,0.3,0.469736,0.510962,0.534374,0.861448,0.147805,0.230257,0.277082,0.722896,0.208333,2.979167
1,Single-TB,10,0.3,0.475382,0.519027,0.561204,0.862115,0.159098,0.246388,0.330742,0.724231,0.208333,2.958333


## Optional: U-Net Positive/Negative Test Metrics

The U-Net baseline CSV only stores overall test metrics. Run the next cell if you want to recompute saved U-Net+ResNet34 checkpoints on the same test splits and add:

- `positive_iou`
- `positive_f1`
- `negative_fp_rate`

This makes the U-Net rows easier to compare with SAM3 zero-shot and few-shot. The cell requires the original baseline environment (`torch`, `fastai`, `segmentation_models_pytorch`, and `PIL`). If those packages are unavailable, skip this cell; the final comparison will still run, but U-Net positive-only columns will remain empty.


In [13]:
RUN_UNET_POSITIVE_NEGATIVE_METRICS = True
UNET_MODEL_DIR = PROJECT_ROOT / "CSCI5527-final" / "baseline_models" / "models"
UNET_DETAIL_CSV = OUTPUT_DIR / "unet_resnet34_imagenet_positive_negative_test_metrics.csv"
UNET_PER_IMAGE_CSV = OUTPUT_DIR / "unet_resnet34_imagenet_per_image_test_metrics.csv"

if not RUN_UNET_POSITIVE_NEGATIVE_METRICS:
    print("Skipping U-Net positive/negative metric recomputation.")
elif UNET_DETAIL_CSV.exists() and "csv_target" in pd.read_csv(UNET_PER_IMAGE_CSV, nrows=1).columns:
    print("Using existing U-Net positive/negative metric CSV:", UNET_DETAIL_CSV)
    display(pd.read_csv(UNET_DETAIL_CSV))
else:
    import sys
    sys.path.insert(0, str(PROJECT_ROOT / "CSCI5527-final"))

    try:
        import torch
        import segmentation_models_pytorch as smp
        from preprocessing import TunnelDataPipeline
    except Exception as exc:
        print("Could not import the baseline inference dependencies.")
        print("Install/use the baseline environment, then rerun this cell if you need U-Net positive-only metrics.")
        print("Import error:", repr(exc))
    else:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print("Using device:", device)

        baseline_experiments = {
            "Single-TB": {
                "train_files": ["TB_train.csv"],
                "val_files": ["TB_val.csv"],
                "test_files": ["TB_test.csv"],
            },
            "Shift-TA_TC-to-TB_10pct": {
                "train_files": ["TA_train.csv", "TC_train.csv"],
                "val_files": ["TA_val.csv", "TC_val.csv"],
                "test_files": ["TB_test.csv"],
            },
            "Shift-TA_TB-to-TC_10pct": {
                "train_files": ["TA_train.csv", "TB_train.csv"],
                "val_files": ["TA_val.csv", "TB_val.csv"],
                "test_files": ["TC_test.csv"],
            },
        }

        def build_baseline_test_loader(config):
            dataset_folder = PROJECT_ROOT / "CSCI5527-final" / "TACK_Tunnel_Data"
            csv_source_dir = dataset_folder / "2_model_input"
            raw_mask_dir = dataset_folder / "3_mask"
            pipeline = TunnelDataPipeline(base_dir=str(dataset_folder), original_mask_dir=str(raw_mask_dir))
            df_train_val, df_test = pipeline.load_csv_data(
                csv_source_dir=str(csv_source_dir),
                train_files=config["train_files"],
                val_files=config["val_files"],
                test_files=config["test_files"],
            )
            df_train_val_ready = pipeline.sanitize_masks(df_train_val, class_pixel_value=40)
            df_test_ready = pipeline.sanitize_masks(df_test, class_pixel_value=40)
            _, _, test_dl = pipeline.get_dataloaders(
                train_val_df=df_train_val_ready,
                test_df=df_test_ready,
                bs=16,
                img_size=512,
                custom_stats=None,
            )
            # Keep original CSV labels for positive/negative grouping. This avoids
            # stale/nonzero sanitized masks causing no_crack images to be counted as crack.
            test_targets = df_test_ready["target"].astype(int).tolist()
            test_filenames = df_test_ready["filename"].astype(str).tolist()
            return test_dl, test_targets, test_filenames

        def per_sample_pixel_stats(pred_mask, true_mask, smooth=1e-6):
            # fastai returns TensorImage/TensorMask subclasses; convert them back to
            # base torch.Tensor before torch logical ops to avoid __torch_function__ dispatch issues.
            pred = pred_mask.as_subclass(torch.Tensor).bool()
            true = true_mask.as_subclass(torch.Tensor).bool()
            tp = torch.logical_and(pred, true).sum().float().item()
            fp = torch.logical_and(pred, torch.logical_not(true)).sum().float().item()
            fn = torch.logical_and(torch.logical_not(pred), true).sum().float().item()
            tn = torch.logical_and(torch.logical_not(pred), torch.logical_not(true)).sum().float().item()
            iou = (tp + smooth) / (tp + fp + fn + smooth)
            precision = (tp + smooth) / (tp + fp + smooth)
            recall = (tp + smooth) / (tp + fn + smooth)
            f1 = 2.0 * precision * recall / (precision + recall + smooth)
            return {
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "tn": tn,
                "iou": iou,
                "f1": f1,
                "precision": precision,
                "recall": recall,
            }

        summary_rows = []
        per_image_rows = []

        for experiment_name, config in baseline_experiments.items():
            model_run = f"Unet-resnet34-imagenet_{experiment_name}"
            ckpt_path = UNET_MODEL_DIR / f"{model_run}.pth"
            print("\n" + "=" * 100)
            print("Evaluating:", model_run)
            print("Checkpoint:", ckpt_path)

            if not ckpt_path.exists():
                print("Skipping: checkpoint not found.")
                continue

            test_dl, test_targets, test_filenames = build_baseline_test_loader(config)
            model = smp.Unet("resnet34", encoder_weights="imagenet", classes=2)
            state = torch.load(ckpt_path, map_location=device)
            model.load_state_dict(state)
            model.to(device)
            model.eval()

            image_index = 0
            with torch.no_grad():
                for images, masks in test_dl:
                    images = torch.as_tensor(images, device=device)
                    masks = torch.as_tensor(masks, device=device).long()
                    if masks.ndim == 4:
                        masks = masks.squeeze(1)
                    outputs = model(images)
                    preds = outputs.argmax(dim=1)

                    for batch_idx in range(preds.shape[0]):
                        pred_mask = preds[batch_idx].detach().cpu()
                        true_mask = masks[batch_idx].detach().cpu()
                        true_base = true_mask.as_subclass(torch.Tensor)
                        pred_base = pred_mask.as_subclass(torch.Tensor)
                        csv_target = int(test_targets[image_index])
                        file_name = test_filenames[image_index]

                        # Use the original CSV target as the source of truth for
                        # positive/negative grouping. For target=0, force GT to empty
                        # so U-Net and SAM3 evaluate no_crack images consistently.
                        if csv_target == 0:
                            true_base = torch.zeros_like(true_base)

                        stats = per_sample_pixel_stats(pred_base == 1, true_base == 1)
                        gt_pixels = int((true_base == 1).sum().item())
                        pred_pixels = int((pred_base == 1).sum().item())
                        row = {
                            "model": "U-Net+ResNet34 ImageNet",
                            "model_run": model_run,
                            "experiment_name": experiment_name,
                            "image_index": image_index,
                            "file_name": file_name,
                            "csv_target": csv_target,
                            "has_gt_crack": csv_target == 1,
                            "has_prediction": pred_pixels > 0,
                            "gt_pixels": gt_pixels,
                            "pred_pixels": pred_pixels,
                            **stats,
                        }
                        per_image_rows.append(row)
                        image_index += 1

            exp_df = pd.DataFrame([r for r in per_image_rows if r["model_run"] == model_run])
            positive = exp_df[exp_df["has_gt_crack"]]
            negative = exp_df[~exp_df["has_gt_crack"]]
            summary = {
                "model": "U-Net+ResNet34 ImageNet",
                "model_run": model_run,
                "experiment_name": experiment_name,
                "num_test": int(len(exp_df)),
                "num_positive": int(len(positive)),
                "num_negative": int(len(negative)),
                "positive_iou": float(positive["iou"].mean()) if len(positive) else np.nan,
                "positive_f1": float(positive["f1"].mean()) if len(positive) else np.nan,
                "positive_precision": float(positive["precision"].mean()) if len(positive) else np.nan,
                "positive_recall": float(positive["recall"].mean()) if len(positive) else np.nan,
                "negative_fp_rate": float((negative["pred_pixels"] > 0).mean()) if len(negative) else np.nan,
                "negative_clean_rate": float((negative["pred_pixels"] == 0).mean()) if len(negative) else np.nan,
                "per_image_test_iou": float(exp_df["iou"].mean()),
                "per_image_test_f1": float(exp_df["f1"].mean()),
            }
            summary_rows.append(summary)
            print(
                f"positive_iou={summary['positive_iou']:.4f}, "
                f"positive_f1={summary['positive_f1']:.4f}, "
                f"negative_fp_rate={summary['negative_fp_rate']:.4f}"
            )

        unet_detail_df = pd.DataFrame(summary_rows)
        unet_per_image_df = pd.DataFrame(per_image_rows)
        unet_detail_df.to_csv(UNET_DETAIL_CSV, index=False)
        unet_per_image_df.to_csv(UNET_PER_IMAGE_CSV, index=False)
        print("\nSaved U-Net summary:", UNET_DETAIL_CSV)
        print("Saved U-Net per-image metrics:", UNET_PER_IMAGE_CSV)
        display(unet_detail_df)


Using existing U-Net positive/negative metric CSV: /users/7/yu001011/csci5527/CSCI5527-final/SAM3/outputs/fewshot_metric_conversion/unet_resnet34_imagenet_positive_negative_test_metrics.csv


,model,model_run,experiment_name,num_test,num_positive,num_negative,positive_iou,positive_f1,positive_precision,positive_recall,negative_fp_rate,negative_clean_rate,per_image_test_iou,per_image_test_f1
0,U-Net+ResNet34 ImageNet,Unet-resnet34-imagenet_Single-TB,Single-TB,48,24,24,0.279395,0.390888,0.553532,0.404237,0.416667,0.583333,0.431364,0.487111
1,U-Net+ResNet34 ImageNet,Unet-resnet34-imagenet_Shift-TA_TC-to-TB_10pct,Shift-TA_TC-to-TB_10pct,48,24,24,0.302425,0.421722,0.493254,0.493662,0.583333,0.416667,0.359546,0.419194
2,U-Net+ResNet34 ImageNet,Unet-resnet34-imagenet_Shift-TA_TB-to-TC_10pct,Shift-TA_TB-to-TC_10pct,38,19,19,0.386960,0.509815,0.634084,0.578746,0.315789,0.684211,0.535586,0.597013


## Compare With U-Net Baseline and SAM3 Zero-Shot

This optional cell loads existing U-Net/ResNet34 baseline results and SAM3 zero-shot results, then combines them with the SAM3 few-shot conversion table.

Zero-shot rows are read from `outputs/zero_shot/sam3_zero_shot_batch_baseline_compatible_summary.csv`. Positive-only IoU/F1 and negative false-positive rate are computed from each zero-shot per-image CSV when available.


In [14]:
BASELINE_CSV = PROJECT_ROOT / "CSCI5527-final" / "baseline_models" / "TTD_baseline_results.csv"
UNET_DETAIL_CSV = OUTPUT_DIR / "unet_resnet34_imagenet_positive_negative_test_metrics.csv"
ZERO_SHOT_SUMMARY_CSV = SAM3_WORK_ROOT / "outputs" / "zero_shot" / "sam3_zero_shot_batch_baseline_compatible_summary.csv"

wanted = ["Single-TB", "Shift-TA_TC-to-TB_10pct", "Shift-TA_TB-to-TC_10pct"]
compare_cols = [
    "model",
    "model_run",
    "experiment_name",
    "shot_per_class",
    "score_threshold",
    "test_iou",
    "test_f1",
    "test_precision",
    "test_recall",
    "positive_iou",
    "positive_f1",
    "negative_fp_rate",
]

compare_frames = []

# U-Net / ResNet34 baseline.
if not BASELINE_CSV.exists():
    print("Baseline CSV not found:", BASELINE_CSV)
else:
    baseline_df = pd.read_csv(BASELINE_CSV)
    baseline_df["experiment_name"] = baseline_df["Experiment"].str.replace("Unet-resnet34-imagenet_", "", regex=False)
    baseline_subset = baseline_df[baseline_df["experiment_name"].isin(wanted)].copy()
    baseline_subset = baseline_subset.rename(columns={
        "Experiment": "model_run",
        "Test_IoU": "test_iou",
        "Test_F1": "test_f1",
        "Test_Prec": "test_precision",
        "Test_Recall": "test_recall",
    })
    baseline_subset["model"] = "U-Net+ResNet34 ImageNet"
    baseline_subset["shot_per_class"] = "full baseline"
    baseline_subset["score_threshold"] = np.nan
    baseline_subset["positive_iou"] = np.nan
    baseline_subset["positive_f1"] = np.nan
    baseline_subset["negative_fp_rate"] = np.nan

    if UNET_DETAIL_CSV.exists():
        unet_detail = pd.read_csv(UNET_DETAIL_CSV)
        fill_cols = ["experiment_name", "positive_iou", "positive_f1", "negative_fp_rate"]
        baseline_subset = baseline_subset.drop(columns=["positive_iou", "positive_f1", "negative_fp_rate"]).merge(
            unet_detail[fill_cols],
            on="experiment_name",
            how="left",
        )
    else:
        print("U-Net positive/negative detail CSV not found; U-Net positive-only columns will be empty:", UNET_DETAIL_CSV)

    compare_frames.append(baseline_subset[compare_cols])

# SAM3 zero-shot baseline. Per-image CSVs let us add positive-only and negative-FP summaries.
def summarize_zero_shot_per_image(per_image_csv):
    if not isinstance(per_image_csv, str) or not per_image_csv:
        return {"positive_iou": np.nan, "positive_f1": np.nan, "negative_fp_rate": np.nan}
    path = Path(per_image_csv)
    if not path.exists():
        return {"positive_iou": np.nan, "positive_f1": np.nan, "negative_fp_rate": np.nan}

    per_image = pd.read_csv(path)
    if "target" in per_image.columns:
        positive = per_image[per_image["target"].astype(int) == 1]
        negative = per_image[per_image["target"].astype(int) == 0]
    elif "label" in per_image.columns:
        positive = per_image[per_image["label"].astype(str) == "crack"]
        negative = per_image[per_image["label"].astype(str) != "crack"]
    else:
        return {"positive_iou": np.nan, "positive_f1": np.nan, "negative_fp_rate": np.nan}

    # For empty-GT negative images, the existing metric code gives IoU ~= 1 when
    # the prediction is also empty. Treat noticeably lower IoU as a false positive.
    negative_fp_rate = float((negative["iou"] < 0.999).mean()) if len(negative) else np.nan
    return {
        "positive_iou": float(positive["iou"].mean()) if len(positive) else np.nan,
        "positive_f1": float(positive["f1"].mean()) if len(positive) else np.nan,
        "negative_fp_rate": negative_fp_rate,
    }

if not ZERO_SHOT_SUMMARY_CSV.exists():
    print("Zero-shot summary CSV not found:", ZERO_SHOT_SUMMARY_CSV)
else:
    zero_df = pd.read_csv(ZERO_SHOT_SUMMARY_CSV)
    zero_df = zero_df[zero_df["Experiment"].isin(wanted)].copy()
    zero_extra = zero_df["Per_Image_Results_CSV"].apply(summarize_zero_shot_per_image).apply(pd.Series)
    zero_df = pd.concat([zero_df.reset_index(drop=True), zero_extra.reset_index(drop=True)], axis=1)
    zero_df = zero_df.rename(columns={
        "Experiment": "experiment_name",
        "Test_IoU": "test_iou",
        "Test_F1": "test_f1",
        "Test_Prec": "test_precision",
        "Test_Recall": "test_recall",
        "ConfidenceThreshold": "score_threshold",
    })
    zero_df["model"] = "SAM3 zero-shot"
    zero_df["model_run"] = zero_df["Model"] + "_" + zero_df["experiment_name"]
    zero_df["shot_per_class"] = "zero-shot"
    compare_frames.append(zero_df[compare_cols])

# SAM3 few-shot converted metrics. This is the exploratory best-threshold view.
if "best_by_positive_f1" not in globals() or best_by_positive_f1.empty:
    print("Few-shot best-threshold table is not available. Run the previous cell first.")
else:
    sam3_compare = best_by_positive_f1.copy()
    sam3_compare["model"] = "SAM3 few-shot"
    sam3_compare["model_run"] = sam3_compare["experiment_key"]
    compare_frames.append(sam3_compare[compare_cols])

if not compare_frames:
    raise RuntimeError("No comparison sources were available.")

comparison_df = pd.concat(compare_frames, ignore_index=True)
comparison_csv = OUTPUT_DIR / "sam3_vs_unet_pixel_metric_comparison.csv"
comparison_df.to_csv(comparison_csv, index=False)
print("Saved comparison table:", comparison_csv)

display(comparison_df.sort_values(["experiment_name", "model", "shot_per_class"]))


Saved comparison table: /users/7/yu001011/csci5527/CSCI5527-final/SAM3/outputs/fewshot_metric_conversion/sam3_vs_unet_pixel_metric_comparison.csv


,model,model_run,experiment_name,shot_per_class,score_threshold,test_iou,test_f1,test_precision,test_recall,positive_iou,positive_f1,negative_fp_rate
6,SAM3 few-shot,shift_ta_tb_to_tc_10pct_5shot_stable_lowlr_v1,Shift-TA_TB-to-TC_10pct,5,0.30,0.549616,0.575344,0.876896,0.640908,0.151863,0.203320,0.052632
7,SAM3 few-shot,shift_ta_tb_to_tc_10pct_10shot_stable_lowlr_v1,Shift-TA_TB-to-TC_10pct,10,0.30,0.551037,0.576743,0.881234,0.635971,0.154706,0.206117,0.052632
8,SAM3 few-shot,shift_ta_tb_to_tc_10pct_25shot_stable_lowlr_v1,Shift-TA_TB-to-TC_10pct,25,0.30,0.525805,0.546028,0.914113,0.578445,0.104241,0.144688,0.052632
9,SAM3 few-shot,shift_ta_tb_to_tc_10pct_50shot_stable_lowlr_v1,Shift-TA_TB-to-TC_10pct,50,0.30,0.550985,0.571756,0.944274,0.572398,0.101970,0.143512,0.000000
5,SAM3 zero-shot,SAM3_zero_shot_Shift-TA_TB-to-TC_10pct,Shift-TA_TB-to-TC_10pct,zero-shot,0.05,0.199160,0.203327,0.304480,0.527431,0.029899,0.038233,0.631579
1,U-Net+ResNet34 ImageNet,Unet-resnet34-imagenet_Shift-TA_TB-to-TC_10pct,Shift-TA_TB-to-TC_10pct,full baseline,NaN,0.380002,0.549345,0.506741,0.599940,0.386960,0.509815,0.315789
10,SAM3 few-shot,shift_ta_tc_to_tb_10pct_5shot_stable_lowlr_v1,Shift-TA_TC-to-TB_10pct,5,0.30,0.470343,0.512004,0.534993,0.861391,0.149020,0.232342,0.208333
11,SAM3 few-shot,shift_ta_tc_to_tb_10pct_10shot_stable_lowlr_v1,Shift-TA_TC-to-TB_10pct,10,0.30,0.475069,0.518272,0.540193,0.861744,0.158470,0.244877,0.208333
12,SAM3 few-shot,shift_ta_tc_to_tb_10pct_25shot_stable_lowlr_v1,Shift-TA_TC-to-TB_10pct,25,0.30,0.522319,0.566001,0.629740,0.837142,0.169639,0.257002,0.125000
13,SAM3 few-shot,shift_ta_tc_to_tb_10pct_50shot_stable_lowlr_v1,Shift-TA_TC-to-TB_10pct,50,0.30,0.536817,0.585902,0.668467,0.826053,0.198634,0.296804,0.125000
